In [1]:
# ① 설정과 데이터 읽기: 공유 폴더의 표 3개를 읽고 환자·분할·창 목록을 합침
from pathlib import Path
import time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score

DATA_DIR = Path("/mnt/c/data/music/SCD_Holter_DL_10s")   # 공유 폴더 (윈도우: C:\data\music\SCD_Holter_DL_10s)
SEED = 42
TRAIN_WINDOWS_PER_PATIENT = 10_000   # 학습: 환자당 창 수 (최대 8,634개라 10,000이면 전체)
VAL_WINDOWS_PER_PATIENT = 10_000     # 검증: 환자당 창 수 (10,000이면 전체)
EPOCHS = 10
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
NUM_WORKERS = 4                   # 데이터를 동시에 읽는 작업 수

torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

patients = pd.read_csv(DATA_DIR / "index" / "patients.csv").merge(
    pd.read_csv(DATA_DIR / "index" / "split_team_v1.csv"), on="pid")
windows = pd.read_csv(DATA_DIR / "index" / "windows.csv.gz", usecols=["pid", "row"])
windows = windows.merge(patients[["pid", "label_scd", "split"]], on="pid")

display(pd.crosstab(patients["split"], patients["label_scd"], margins=True))
print(f"창 {len(windows):,}개 · 장치 {device}")


label_scd,0,1,All
split,,,
test,116,18,134
train,406,54,460
validation,99,15,114
All,621,87,708


창 5,868,655개 · 장치 cuda


In [2]:
# ② 창 뽑기와 데이터셋: 환자마다 창을 n개씩 무작위로 뽑고, 파일에서 그 창만 읽어 정규화
def sample_windows(split_name, n_per_patient, seed):
    """한 분할에서 환자마다 창을 최대 n개씩 무작위로 뽑음"""
    part = windows[windows["split"] == split_name]
    return part.sample(frac=1, random_state=seed).groupby("pid").head(n_per_patient).reset_index(drop=True)


class HolterWindows(Dataset):
    """창 목록 한 줄 → (파형 (3, 2000) float32, 정답)"""

    def __init__(self, table):
        self.pids = table["pid"].to_numpy()
        self.rows = table["row"].to_numpy()
        self.labels = table["label_scd"].to_numpy(dtype=np.float32)
        self._files = {}                      # 환자 파일은 처음 쓸 때 한 번만 연다 (필요한 창만 읽음)

    def __len__(self):
        return len(self.pids)

    def _file(self, pid):
        if pid not in self._files:
            self._files[pid] = np.load(DATA_DIR / "windows" / f"{pid}.npy", mmap_mode="r")
        return self._files[pid]

    def __getitem__(self, i):
        x = self._file(self.pids[i])[self.rows[i]].astype(np.float32)
        x = (x - x.mean(axis=1, keepdims=True)) / (x.std(axis=1, keepdims=True) + 1e-6)   # 유도별 표준화 (0인 Z는 그대로 0)
        return torch.from_numpy(x), torch.tensor(self.labels[i])


val_table = sample_windows("validation", VAL_WINDOWS_PER_PATIENT, SEED)
val_loader = DataLoader(HolterWindows(val_table), batch_size=512, shuffle=False, num_workers=NUM_WORKERS)
print(f"검증 창 {len(val_table):,}개 · 검증 환자 {val_table['pid'].nunique()}명")


검증 창 11,400개 · 검증 환자 114명


In [3]:
# ③ 모델: 가장 기본적인 1D CNN (합성곱 4층 → 전체 평균 → 출력 1개)
def conv_block(c_in, c_out):
    """합성곱 → 배치 정규화 → ReLU → 길이를 절반으로"""
    return torch.nn.Sequential(
        torch.nn.Conv1d(c_in, c_out, kernel_size=7, padding=3),
        torch.nn.BatchNorm1d(c_out),
        torch.nn.ReLU(),
        torch.nn.MaxPool1d(2))


class SimpleCNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.features = torch.nn.Sequential(
            conv_block(3, 32), conv_block(32, 64), conv_block(64, 128), conv_block(128, 128))
        self.head = torch.nn.Sequential(torch.nn.AdaptiveAvgPool1d(1), torch.nn.Flatten(),
                                        torch.nn.Dropout(0.3), torch.nn.Linear(128, 1))

    def forward(self, x):
        return self.head(self.features(x)).squeeze(1)      # SCD일 정도 (확률로 바꾸기 전 값)


model = SimpleCNN().to(device)
print(f"모델 크기: {sum(p.numel() for p in model.parameters()):,}개 값")


모델 크기: 188,225개 값


In [4]:
# ④ 학습과 검증: 에폭마다 학습 창을 새로 뽑아 학습 → 검증 환자 단위 AUROC · AP → 가장 좋은 모델 저장
SAVE_PATH = Path("../results/05_dl_full_best.pt")          # 전체 데이터 실행용 (100개 실행 결과는 05_dl_baseline_best.pt에 그대로 둠)
HISTORY_PATH = Path("../results/05_dl_full_history.csv")
SAVE_PATH.parent.mkdir(exist_ok=True)

n_pos = patients.loc[patients["split"] == "train", "label_scd"].sum()
n_neg = (patients["split"] == "train").sum() - n_pos
loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(n_neg / n_pos, device=device))   # SCD가 적은 것 보정
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)


def train_one_epoch(epoch):
    """학습 환자에게서 창을 새로 뽑아 한 바퀴 학습 → 평균 손실"""
    table = sample_windows("train", TRAIN_WINDOWS_PER_PATIENT, SEED + epoch)
    loader = DataLoader(HolterWindows(table), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    model.train()
    losses = []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        loss = loss_fn(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return np.mean(losses)


def evaluate_patients():
    """검증 창 예측 → 환자별 평균 → (AUROC, AP)"""
    model.eval()
    probs = []
    with torch.no_grad():
        for x, _ in val_loader:
            probs.append(torch.sigmoid(model(x.to(device))).cpu())
    per_patient = val_table.assign(prob=torch.cat(probs).numpy()).groupby("pid").agg(
        prob=("prob", "mean"), label=("label_scd", "first"))
    return roc_auc_score(per_patient["label"], per_patient["prob"]), average_precision_score(per_patient["label"], per_patient["prob"])


history, best_auroc = [], 0.0
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    loss = train_one_epoch(epoch)
    auroc, ap = evaluate_patients()
    history.append({"epoch": epoch, "loss": loss, "val_AUROC": auroc, "val_AP": ap})
    if auroc > best_auroc:
        best_auroc = auroc
        torch.save(model.state_dict(), SAVE_PATH)
    print(f"에폭 {epoch:2d} · 손실 {loss:.3f} · 검증 AUROC {auroc:.3f} · AP {ap:.3f} · {time.time() - t0:.0f}초")
    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)   # 중간에 끊겨도 끝난 에폭 기록은 남음

history = pd.DataFrame(history)
print(f"\n가장 좋은 검증 AUROC {best_auroc:.3f} (저장: {SAVE_PATH})")
print(f"참고: 아무것도 안 배운 모델의 AP ≈ SCD 비율 {patients.loc[patients['split'] == 'validation', 'label_scd'].mean():.3f}")


에폭  1 · 손실 0.614 · 검증 AUROC 0.615 · AP 0.206 · 12초
에폭  2 · 손실 0.241 · 검증 AUROC 0.560 · AP 0.171 · 11초
에폭  3 · 손실 0.125 · 검증 AUROC 0.565 · AP 0.195 · 11초
에폭  4 · 손실 0.095 · 검증 AUROC 0.549 · AP 0.178 · 11초
에폭  5 · 손실 0.083 · 검증 AUROC 0.593 · AP 0.218 · 12초
에폭  6 · 손실 0.076 · 검증 AUROC 0.558 · AP 0.186 · 11초
에폭  7 · 손실 0.055 · 검증 AUROC 0.594 · AP 0.223 · 12초
에폭  8 · 손실 0.055 · 검증 AUROC 0.616 · AP 0.194 · 11초
에폭  9 · 손실 0.062 · 검증 AUROC 0.525 · AP 0.156 · 11초
에폭 10 · 손실 0.044 · 검증 AUROC 0.605 · AP 0.239 · 11초

가장 좋은 검증 AUROC 0.616 (저장: ../results/05_dl_baseline_best.pt)
참고: 아무것도 안 배운 모델의 AP ≈ SCD 비율 0.132
